# Hybrid Search (BM25 + semantic)

> Notebook generated from [`examples/rag/05_hybrid_search.py`](../../examples/rag/05_hybrid_search.py).

| Data | Value |
|------|-------|
| **Dataset** | AG News (embedded, classified news) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Combined score `α·semantic + (1-α)·BM25`; sweep over α=0.0/0.3/0.5/0.7/1.0.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Hybrid Search — Lexical BM25 + semantic with weighted fusion
=============================================================
Architecture: SPEC-RAG-004 / prismal.rag.hybrid

Dataset: AG News (News Topic Classification)
  • 127,600 news articles in 4 categories: World, Sports, Business, Science/Tech.
  • Reference: https://huggingface.co/datasets/fancyzhx/ag_news
  • Why: Hybrid Search combines BM25 (exact lexical match) with
    semantic search. News articles have technical terms and proper nouns
    that BM25 handles well (exact match), and thematic topics that semantics
    captures better. AG News is the standard benchmark for news retrieval.

Hybrid Search architecture description:
  Formula: score(d) = α × semantic(d) + (1 - α) × bm25_norm(d)
  - BM25 (Okapi): probabilistic TF-IDF, excellent for exact keywords
  - Semantic: dense embeddings, excellent for synonyms and paraphrases
  - α = 0.5 (default): equal balance between both
  - α = 1.0: semantic only
  - α = 0.0: BM25 only

  Limitation: in-memory BM25Okapi; >100K docs → use a backend with an inverted index.

Included experiments:
  - Comparison α=0.0, 0.3, 0.5, 0.7, 1.0 on the same corpus
  - Cases where BM25 wins (exact keywords, proper nouns)
  - Cases where semantic wins (synonyms, paraphrases)

Usage:
    uv run python examples/rag/05_hybrid_search.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
from typing import TYPE_CHECKING

from langchain_core.documents import Document

from prismal.rag.hybrid import HybridSearchEngine
from prismal.rag.vector_store import ChromaVectorStore

if TYPE_CHECKING:
    from prismal.rag.crag import RetrievedChunk

## Dataset: AG News articles

In [ ]:
# Representative sample from 4 AG News categories.
AG_NEWS_ARTICLES = [
    # Category: Science/Technology
    {
        "id": "ANG001",
        "category": "Science/Tech",
        "title": "GPT-4 Achieves Human-Level Performance on Medical Licensing Exams",
        "text": (
            "OpenAI's GPT-4 language model has demonstrated human-level performance on "
            "the United States Medical Licensing Examination (USMLE), scoring above the "
            "passing threshold. The model achieves 86.7% accuracy on Step 1 questions "
            "without any medical fine-tuning. Researchers attribute this to GPT-4's "
            "broad pretraining on medical literature, textbooks, and clinical guidelines."
        ),
    },
    {
        "id": "ANG002",
        "category": "Science/Tech",
        "title": "Quantum Computing Milestone: IBM Reaches 1000-Qubit Processor",
        "text": (
            "IBM has unveiled its Condor quantum processor with 1,121 qubits, surpassing "
            "the 1000-qubit milestone. The processor uses superconducting transmon qubits "
            "operating at temperatures near absolute zero. IBM claims this represents a "
            "major step toward quantum advantage in optimization and simulation problems. "
            "The system achieves error rates below 0.3% per two-qubit gate operation."
        ),
    },
    # Category: Business
    {
        "id": "ANG003",
        "category": "Business",
        "title": "Microsoft Acquires AI Startup for $1.5 Billion in Strategic Move",
        "text": (
            "Microsoft Corporation announced the acquisition of an AI startup specializing "
            "in enterprise language models for $1.5 billion. The deal includes integration "
            "with Azure OpenAI Service. Analysts expect the acquisition to boost Microsoft's "
            "market share in enterprise AI solutions by 15% over the next fiscal year. "
            "The startup's 200 engineers will join Microsoft's AI division in Redmond."
        ),
    },
    {
        "id": "ANG004",
        "category": "Business",
        "title": "NVIDIA Stock Surges 40% as AI Chip Demand Exceeds Projections",
        "text": (
            "NVIDIA Corporation shares surged 40% in after-hours trading following "
            "quarterly results that showed GPU demand far exceeding Wall Street projections. "
            "Data center revenue reached $18.4 billion, driven by AI training workloads. "
            "CEO Jensen Huang stated that the company is accelerating H100 production "
            "to meet unprecedented demand from cloud providers and AI research labs."
        ),
    },
    # Category: Sports
    {
        "id": "ANG005",
        "category": "Sports",
        "title": "Barcelona FC Signs AI-Powered Training System for Player Performance",
        "text": (
            "FC Barcelona has implemented an AI-powered player performance analytics "
            "system developed by a Silicon Valley startup. The system uses computer vision "
            "and biomechanics analysis to optimize training loads and predict injury risk. "
            "The club reports a 23% reduction in soft tissue injuries since implementation. "
            "Manager Xavi Hernandez praised the system for providing data-driven insights."
        ),
    },
    # Category: World
    {
        "id": "ANG006",
        "category": "World",
        "title": "European Union Passes AI Act: Comprehensive Regulation for Artificial Intelligence",
        "text": (
            "The European Parliament has passed the EU AI Act, the world's first comprehensive "
            "legal framework for artificial intelligence. The regulation classifies AI systems "
            "by risk level: unacceptable, high, limited, and minimal risk. High-risk systems "
            "in healthcare, law enforcement, and critical infrastructure face strict compliance "
            "requirements. The Act prohibits real-time biometric surveillance in public spaces "
            "and social scoring systems. Companies face fines up to 7% of global revenue."
        ),
    },
    {
        "id": "ANG007",
        "category": "Science/Tech",
        "title": "LangChain and LlamaIndex Partnership Accelerates Enterprise RAG Adoption",
        "text": (
            "LangChain and LlamaIndex announced a technical partnership to standardize "
            "Retrieval-Augmented Generation (RAG) pipelines for enterprise deployments. "
            "The partnership introduces interoperability between both frameworks' document "
            "loaders, vector stores, and LLM integrations. Early adopters report 60% "
            "reduction in development time for production RAG systems. The collaboration "
            "targets the growing market of companies deploying domain-specific AI assistants."
        ),
    },
    {
        "id": "ANG008",
        "category": "Science/Tech",
        "title": "ChromaDB Releases Version 0.5 with Enhanced Filtering and MMR Support",
        "text": (
            "ChromaDB vector database released version 0.5.0 with improved metadata "
            "filtering, Maximal Marginal Relevance (MMR) support for diverse retrieval, "
            "and 3x faster query performance. The update supports hybrid search combining "
            "dense vector similarity with sparse BM25 retrieval. ChromaDB now handles "
            "collections with over 10 million embeddings efficiently. The release also "
            "includes native integration with LangChain and LlamaIndex frameworks."
        ),
    },
]

## Queries for BM25 vs Semantic comparison

In [ ]:
HYBRID_QUERIES = [
    # BM25 wins: exact keywords, proper nouns, technical terms
    {
        "id": "HQ1",
        "query": "NVIDIA H100 GPU data center revenue Jensen Huang",
        "type": "keyword_exact",
        "expected_source": "ANG004",
        "reason": "Exact proper nouns → BM25 beats semantic",
    },
    {
        "id": "HQ2",
        "query": "ChromaDB version 0.5 MMR metadata filtering BM25",
        "type": "keyword_exact",
        "expected_source": "ANG008",
        "reason": "Exact technical terms + version numbers",
    },
    # Semantic wins: synonyms, paraphrases, concepts
    {
        "id": "HQ3",
        "query": "Government regulation of intelligent systems in Europe",
        "type": "semantic_paraphrase",
        "expected_source": "ANG006",
        "reason": "Paraphrase of 'EU AI Act' — semantic captures the concept",
    },
    {
        "id": "HQ4",
        "query": "athlete performance improved with data analytics",
        "type": "semantic_paraphrase",
        "expected_source": "ANG005",
        "reason": "Paraphrase of 'player performance analytics' — semantic",
    },
    # Hybrid wins: combination of keyword + semantic
    {
        "id": "HQ5",
        "query": "RAG pipeline enterprise LangChain integration performance",
        "type": "hybrid_optimal",
        "expected_source": "ANG007",
        "reason": "Mix of exact keywords (LangChain) + concept (RAG pipeline)",
    },
    {
        "id": "HQ6",
        "query": "Microsoft billion dollar AI startup Azure cloud services",
        "type": "hybrid_optimal",
        "expected_source": "ANG003",
        "reason": "Proper nouns (Microsoft) + concept (cloud AI services)",
    },
]


async def setup_hybrid(alpha: float = 0.5) -> HybridSearchEngine:
    """Set up the Hybrid Search engine with the AG News corpus.

    Args:
        alpha: Weight of the semantic component [0.0=BM25, 0.5=hybrid, 1.0=semantic].

    Returns:
        Configured HybridSearchEngine ready for search.
    """
    store = ChromaVectorStore(collection_name=f"ag_news_hybrid_{alpha:.1f}")

    # Create documents
    docs = [
        Document(
            page_content=f"{art['title']}. {art['text']}",
            metadata={
                "source": art["id"],
                "category": art["category"],
                "title": art["title"],
                "chunk_id": art["id"],
                "dataset": "AG_News",
            },
        )
        for art in AG_NEWS_ARTICLES
    ]

    store.add_documents(docs)

    engine = HybridSearchEngine(
        vector_store=store,
        alpha=alpha,
    )

    # Build BM25 index with the same corpus
    corpus = [f"{art['title']}. {art['text']}" for art in AG_NEWS_ARTICLES]
    doc_ids = [art["id"] for art in AG_NEWS_ARTICLES]
    engine.build_index(corpus=corpus, doc_ids=doc_ids)

    return engine


def evaluate_retrieval(
    query_info: dict,
    chunks: list[RetrievedChunk],
    top_k: int = 3,
) -> bool:
    """Evaluate whether the expected document is in the top-k results."""
    expected = query_info["expected_source"]
    retrieved_sources = [c.source for c in chunks[:top_k]]
    return expected in retrieved_sources


async def run_alpha_comparison(query_info: dict) -> dict[float, bool]:
    """Compare results across different α values for a query."""
    alphas = [0.0, 0.3, 0.5, 0.7, 1.0]
    results = {}

    for alpha in alphas:
        engine = await setup_hybrid(alpha)
        chunks = engine.search(query_info["query"], k=3)
        results[alpha] = evaluate_retrieval(query_info, chunks)

    return results


async def main() -> None:
    print("=" * 70)
    print("  Hybrid Search (BM25 + Semantic) — Dataset: AG News")
    print("=" * 70)

    print("\n[Hybrid Search formula]")
    print("  score(d) = α × semantic(d) + (1-α) × bm25_norm(d)")
    print("  α = 0.0 → BM25 only (exact lexical)")
    print("  α = 0.5 → equal balance (recommended default)")
    print("  α = 1.0 → semantic only (dense embeddings)")

    # Configure main engine (α=0.5)
    print("\n[Initialization with α=0.5]")
    engine = await setup_hybrid(alpha=0.5)
    print(f"  ✓ BM25 index and vector store ready ({len(AG_NEWS_ARTICLES)} articles)")

    # Evaluate queries
    print(f"\n[Evaluating {len(HYBRID_QUERIES)} AG News queries]")

    for query_info in HYBRID_QUERIES:
        chunks = engine.search(query_info["query"], k=3)
        found = evaluate_retrieval(query_info, chunks, top_k=3)

        print(f"\n[{query_info['id']}] Type: {query_info['type']}")
        print(f"  Query   : {query_info['query']}")
        print(f"  Reason  : {query_info['reason']}")
        print("  Top-3 results:")
        for chunk in chunks[:3]:
            mark = "→" if chunk.source == query_info["expected_source"] else " "
            print(f"    {mark} [{chunk.relevance_score:.3f}] {chunk.source}")
        print(f"  Expected source found: {'✓' if found else '✗'}")

    # Experiment: α comparison
    print("\n[Experiment: effect of the α parameter]")
    print("  Comparing α on 3 representative queries...")
    print()

    sample_queries = HYBRID_QUERIES[:3]  # 1 keyword, 1 semantic, 1 hybrid

    for query_info in sample_queries:
        alpha_results = await run_alpha_comparison(query_info)
        print(f"  [{query_info['id']}] {query_info['type']}")
        print(f"    Query: {query_info['query'][:60]}...")
        result_line = "    " + " | ".join(
            f"α={a:.1f}:{'✓' if ok else '✗'}" for a, ok in alpha_results.items()
        )
        print(result_line)
        print()

    # Conclusions
    print("[Guide for selecting α]")
    guidelines = [
        (0.0, "BM25 only", "Legal documents, source code, exact search"),
        (0.3, "BM25 dominant", "Technical corpora with specialized vocabulary"),
        (0.5, "Balance (default)", "General use — best compromise"),
        (0.7, "Semantic dominant", "Conversational text, frequent synonyms"),
        (1.0, "Semantic only", "Concept search, cross-lingual retrieval"),
    ]
    for alpha, label, use_case in guidelines:
        print(f"  α={alpha:.1f} ({label:20s}): {use_case}")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()